## 04b: Sensitivity Analyses

The four sensitivity analyses, reported at nominal p with no correction for multiple testing.
`D` is computed exactly as in `04a`; `HC` is the non-clinical control group (NCC).

1. **NCC comparison arm** (Tasks 1–2): the phenotype analysis of `04a`, refitted in the HC
   wearable sample.
2. **U-shaped TST** (Task 3): the TST model with an added quadratic term, in MDD and HC.
3. **Representativeness of the Fitbit-eligible MDD subgroup** (Task 4): MDD participants in the
   phenotype analysis compared with the rest of the MDD sample.
4. **Antipsychotic exclusion** (Task 5): the normative-deviation analysis refitted without
   participants with an active antipsychotic prescription. Task 5b compares the excluded and
   retained participants.

Outputs (`results/`): `model_results_hc_sensitivity`, `sleep_duration_ushape_sensitivity`,
`mdd_normative_vs_fitbit_balance`, `antipsychotic_sensitivity` and
`antipsychotic_dropped_vs_retained_balance` (`.parquet`). `04c` adds MDES columns to the first.

## Task 0: Setup

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import percentileofscore, ttest_ind, chi2_contingency, fisher_exact

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *
from src.analysis import (
    ETM_TASKS, TASK_CONFIG, PHENOTYPES, MIN_CELL, RESULT_COLUMNS,
    prepare_cohort, fit_normative_model, normative_sample, fitbit_sample,
    hc_fitbit_sample, phenotype_complete, fit_group_model, residualise_phenotype,
    fit_phenotype_association, result_row, suppressed_row,
)

OVERWRITE = True
RES_OUT = "nb04b"

# Expected number of HC-arm rows (4 phenotypes × 4 tasks)
HC_POOL_SIZE = len(ETM_TASKS) * len(PHENOTYPES)

# ── Load data ─────────────────────────────────────────────────────
cohort = prepare_cohort(load_parquet('cohort/cohort_master.parquet'))
wf     = load_parquet('features/wearable_features.parquet')

print(f'cohort          : {len(cohort):,} rows '
      f'(MDD {(cohort["group"]=="MDD").sum():,} / '
      f'BD {(cohort["group"]=="BD").sum():,} / '
      f'HC {(cohort["group"]=="HC").sum():,})')
print(f'wearable_features: {len(wf):,} rows across {wf["task"].nunique()} tasks')
print(f'OVERWRITE       : {OVERWRITE}')


## Task 1: NCC Comparison Arm

The same residualisation and `D ~ P_z` model as the MDD analysis in `04a`. For HC, `D` is the
standardised residual of the normative model.

In [ ]:
def run_task_hc(task, cohort, wf, hc_rows):
    out_col = TASK_CONFIG[task]['outcome']

    print(f'\n{"─" * 64}')
    print(f'  {task.upper()}  ·  outcome: {out_col}')
    print(f'{"─" * 64}')

    # HC normative model; hc carries D as the in-sample standardised residual.
    m_norm, sigma_hc, hc = fit_normative_model(cohort, task)
    print(f'HC model   N={len(hc):,}  R²={m_norm.rsquared:.3f}  σ_HC={sigma_hc:.4f}')

    hc_c = hc_fitbit_sample(hc, task, wf)
    print(f'Sample Fitbit_HC (sensitivity)  wearable-eligible N={len(hc_c):,}')

    for pheno in PHENOTYPES:
        c_p = phenotype_complete(hc_c, pheno)
        n_p = len(c_p)
        if n_p < MIN_CELL:
            print(f'  Fitbit_HC/{pheno:<18} HC N={n_p} < {MIN_CELL}, suppressing')
            hc_rows.append(suppressed_row(
                task=task, analysis='Fitbit_HC', phenotype=pheno,
                group='HC', n_hc=n_p, n_mdd=None, n_bd=None,
            ))
            continue
        m_c, r_hc, _ = fit_phenotype_association(c_p, task, pheno)
        hc_rows.append(result_row(
            task=task, analysis='Fitbit_HC', phenotype=pheno,
            group='HC', n_hc=n_p, n_mdd=None, n_bd=None,
            beta=m_c.params['P_z'], se=m_c.bse['P_z'],
            t=m_c.tvalues['P_z'],   p=m_c.pvalues['P_z'],
            effect_size=r_hc, r2=r_hc**2,
        ))
        star = ' *' if hc_rows[-1]['p_value'] < 0.05 else ''
        print(f'  Fitbit_HC/{pheno:<18} N={n_p:>5,}  '
              f'β={hc_rows[-1]["beta"]:+.3f}  ES={hc_rows[-1]["effect_size"]:+.3f}  '
              f'p={hc_rows[-1]["p_value"]:.4g}{star}')

    return hc_rows


hc_results = []
for task in ETM_TASKS:
    hc_results = run_task_hc(task, cohort, wf, hc_results)

df = pd.DataFrame(hc_results)

# Local CSV copy: the informative columns only.
cols = [
    'task', 'phenotype', 'n_hc',
    'beta', 'se_hc3', 't_stat', 'p_value',
    'ci_lo', 'ci_hi', 'effect_size', 'r2'
]

df = df[cols]
save_res(RES_OUT, 'sensitivity_hc_cognition-wearable.csv',  OVERWRITE, df,  file_type='csv')

## Task 2: NCC Comparison Arm, Results Table

In [ ]:
assert len(hc_results) == HC_POOL_SIZE, (
    f'Expected {HC_POOL_SIZE} HC sensitivity results (4 phenotypes x 4 tasks); got '
    f'{len(hc_results)}. Sub-20 cells must emit a suppressed placeholder row, not be omitted.'
)

hc_df = pd.DataFrame(hc_results)

# Nominal p, no correction
hc_df['significant'] = hc_df['p_value'] < 0.05

hc_df = hc_df[RESULT_COLUMNS].reset_index(drop=True)

print('\n' + '=' * 100)
print(f'  HC SENSITIVITY SUMMARY  (analysis=Fitbit_HC, nominal p < 0.05, no correction)')
print('=' * 100)
n_supp_hc   = int(hc_df['suppressed'].sum())
n_sig_hc    = int(hc_df['significant'].fillna(False).sum())
n_tested_hc = int((~hc_df['suppressed'].fillna(False)).sum())
print(f'Nominally significant: {n_sig_hc} / {n_tested_hc} non-suppressed tests  '
      f'({n_supp_hc} rows suppressed, n < 20)')

print('\n=== R² summary — rest-activity phenotypes, HC sensitivity arm (Pearson r²) ===')
r2_hc = (
    hc_df[~hc_df['suppressed']]
    .pivot_table(index='phenotype', columns='task', values='r2')
    .round(4)
)
print(r2_hc.to_string())

# Derived output — always saved.
save_parquet(hc_df, 'results/model_results_hc_sensitivity.parquet')


## Task 3: U-Shaped TST

`D ~ P_z + I(P_z ** 2)` for TST (`mean_sleep_duration`), in MDD and HC. A U-shaped (or
inverted-U) association is supported only if the quadratic term is significant, the vertex lies
within the 5th–95th percentile of `P_z`, and predicted performance at both tails is worse than
at the vertex.

In [ ]:
PHENOTYPE = 'mean_sleep_duration'
TAIL_PCTL = 5   # symmetric tails: 5th / 95th percentile of observed P_z

def build_ushape_samples(task, cohort, wf):
    """MDD and HC wearable samples for one task, with TST and photoperiod observed."""
    m_norm, sigma_hc, hc = fit_normative_model(cohort, task)
    mdd   = normative_sample(cohort, task, 'MDD', m_norm, sigma_hc)
    c_mdd = phenotype_complete(fitbit_sample(mdd, task, wf, 'MDD'), PHENOTYPE)
    hc_c  = phenotype_complete(hc_fitbit_sample(hc, task, wf), PHENOTYPE)
    return dict(mdd=c_mdd, hc=hc_c)


def fit_linear_and_quadratic(df, task):
    n = len(df)
    if n < MIN_CELL:
        return dict(n=n, suppressed=True)

    df        = df.copy()
    df['P_z'] = residualise_phenotype(df, task, PHENOTYPE)

    m_lin  = smf.ols('D ~ P_z', data=df).fit(cov_type='HC3')
    m_quad = smf.ols('D ~ P_z + I(P_z ** 2)', data=df).fit(cov_type='HC3')

    b1, b2 = m_quad.params['P_z'], m_quad.params['I(P_z ** 2)']
    p1, p2 = m_quad.pvalues['P_z'], m_quad.pvalues['I(P_z ** 2)']
    se2    = m_quad.bse['I(P_z ** 2)']

    pz_lo = df['P_z'].quantile(TAIL_PCTL / 100)
    pz_hi = df['P_z'].quantile(1 - TAIL_PCTL / 100)

    vertex          = (-b1 / (2 * b2)) if b2 != 0 else np.nan
    vertex_in_range = bool(pz_lo <= vertex <= pz_hi) if np.isfinite(vertex) else False
    vertex_pctl     = float(percentileofscore(df['P_z'], vertex)) if np.isfinite(vertex) else np.nan

    # Predicted D at both tails and at the vertex (clipped to the observed range)
    vertex_for_pred = float(np.clip(vertex, pz_lo, pz_hi)) if np.isfinite(vertex) else 0.0
    pred = m_quad.predict(pd.DataFrame({'P_z': [pz_lo, vertex_for_pred, pz_hi]}))
    d_lo, d_mid, d_hi = pred.iloc[0], pred.iloc[1], pred.iloc[2]

    # Which direction is worse depends on the task
    if TASK_CONFIG[task]['higher_is_better']:
        tails_worse_than_mid = bool((d_lo < d_mid) and (d_hi < d_mid))
    else:
        tails_worse_than_mid = bool((d_lo > d_mid) and (d_hi > d_mid))

    return dict(
        n=n, suppressed=False,
        r_lin=float(np.corrcoef(df['D'], df['P_z'])[0, 1]),
        b_lin=m_lin.params['P_z'], p_lin=m_lin.pvalues['P_z'],
        b1=b1, p1=p1, b2=b2, se2=se2, p2=p2,
        pz_lo=pz_lo, pz_hi=pz_hi,
        vertex=vertex, vertex_in_range=vertex_in_range, vertex_pctl=vertex_pctl,
        d_lo=d_lo, d_mid=d_mid, d_hi=d_hi,
        tails_worse_than_mid=tails_worse_than_mid,
        r2_quad=m_quad.rsquared,
    )


ushape_rows = []
for task in ETM_TASKS:
    wearable = build_ushape_samples(task, cohort, wf)
    for sample_label, df in [('MDD', wearable['mdd']), ('HC', wearable['hc'])]:
        res = fit_linear_and_quadratic(df, task)
        res['task']   = task
        res['sample'] = sample_label
        ushape_rows.append(res)

ushape_df = pd.DataFrame(ushape_rows)


In [ ]:
print('=' * 150)
print('  SLEEP DURATION -- LINEAR vs. QUADRATIC FIT ON D  (nominal p only, no correction; MDD + HC)')
print('=' * 150)
hdr = (f'{"Task":<18}{"Sample":<10}{"N":>6}   {"r_lin":>7}{"p_lin":>9}   '
       f'{"b2":>9}{"se2":>8}{"p2":>9}   {"vertex(Pz)":>11}{"in-range":>9}{"pctl":>7}   '
       f'{"D@lo":>8}{"D@mid":>8}{"D@hi":>8}   {"tails<mid":>10}')
print(hdr)
print('-' * 150)
for _, r in ushape_df.iterrows():
    if r['suppressed']:
        print(f'{r["task"]:<18}{r["sample"]:<10}{r["n"]:>6}   SUPPRESSED (n < 20)')
        continue
    print(f'{r["task"]:<18}{r["sample"]:<10}{r["n"]:>6,.0f}   '
          f'{r["r_lin"]:>+7.3f}{r["p_lin"]:>9.4g}   '
          f'{r["b2"]:>+9.4f}{r["se2"]:>8.4f}{r["p2"]:>9.4g}   '
          f'{r["vertex"]:>+11.3f}{str(r["vertex_in_range"]):>9}{r["vertex_pctl"]:>6.1f}%   '
          f'{r["d_lo"]:>+8.3f}{r["d_mid"]:>+8.3f}{r["d_hi"]:>+8.3f}   '
          f'{str(r["tails_worse_than_mid"]):>10}')

print('\nU-shape supported only if p2 < 0.05, vertex_in_range and tails_worse_than_mid. '
      'Nominal p, no correction.')

save_res(RES_OUT, 'sleep_duration_ushape_sensitivity.csv',  OVERWRITE, ushape_df,  file_type='csv')
save_parquet(ushape_df, 'results/sleep_duration_ushape_sensitivity.parquet')

Columns: `b2`, `p2` quadratic term; `vertex` with its percentile and whether it is in range;
`D@lo`, `D@mid`, `D@hi` predicted `D` at the 5th percentile, the vertex and the 95th percentile.

## Task 4: Representativeness of the Fitbit-Eligible MDD Subgroup

Compares MDD participants included in the phenotype analysis with the rest of the MDD
normative-deviation sample: Welch's t-test and Cohen's d for `D`, age, education, BMI and CCI;
χ² (Fisher's exact test for small expected counts) and phi for sex and device type. A cell is
suppressed if either side has fewer than 20 participants.

In [ ]:
def build_mdd_normative_and_fitbit(task, cohort, wf):
    """MDD Normative sample (with D) and its Fitbit subset for `task`, before any
    per-phenotype completeness filter: the sample that would enter any phenotype model."""
    m_norm, sigma_hc, _ = fit_normative_model(cohort, task)
    ab_mdd = normative_sample(cohort, task, 'MDD', m_norm, sigma_hc)
    c_mdd  = fitbit_sample(ab_mdd, task, wf, 'MDD')
    return ab_mdd, c_mdd


def _continuous_balance_row(task, variable, included_series, excluded_series, n_normative_full):
    x = included_series.dropna()
    y = excluded_series.dropna()
    n_inc, n_exc = len(x), len(y)

    if n_inc < MIN_CELL or n_exc < MIN_CELL:
        return dict(
            task=task, variable=variable, var_type='continuous',
            n_normative_full=n_normative_full, n_included=n_inc, n_excluded=n_exc,
            included_center=None, included_spread=None,
            excluded_center=None, excluded_spread=None,
            test=None, stat=None, p_value=None, effect_size=None,
            significant=None, suppressed=True,
        )

    t_stat, p = ttest_ind(x, y, equal_var=False)
    pooled_sd = np.sqrt(
        ((n_inc - 1) * x.std(ddof=1) ** 2 + (n_exc - 1) * y.std(ddof=1) ** 2)
        / (n_inc + n_exc - 2)
    )
    d = float((x.mean() - y.mean()) / pooled_sd) if pooled_sd > 0 else np.nan

    return dict(
        task=task, variable=variable, var_type='continuous',
        n_normative_full=n_normative_full, n_included=n_inc, n_excluded=n_exc,
        included_center=float(x.mean()), included_spread=float(x.std(ddof=1)),
        excluded_center=float(y.mean()), excluded_spread=float(y.std(ddof=1)),
        test='welch_t', stat=float(t_stat), p_value=float(p), effect_size=d,
        significant=bool(p < 0.05), suppressed=False,
    )


# Category whose proportion is reported for a binary variable. Variables not listed use the
# second sorted category (True for `touch`); without the entry, `sex` would report % male.
BINARY_POS_CATEGORY = {'sex': 'Female'}


def _binary_balance_row(task, variable, included_series, excluded_series, n_normative_full):
    x = included_series.dropna()
    y = excluded_series.dropna()
    n_inc, n_exc = len(x), len(y)

    if n_inc < MIN_CELL or n_exc < MIN_CELL:
        return dict(
            task=task, variable=variable, var_type='binary',
            n_normative_full=n_normative_full, n_included=n_inc, n_excluded=n_exc,
            included_center=None, included_spread=None,
            excluded_center=None, excluded_spread=None,
            test=None, stat=None, p_value=None, effect_size=None,
            significant=None, suppressed=True,
            positive_category=BINARY_POS_CATEGORY.get(variable),
        )

    categories = sorted(set(x.unique()) | set(y.unique()), key=str)
    assert len(categories) == 2, f'{variable} is not binary in this sample: {categories}'
    pos = BINARY_POS_CATEGORY.get(variable, categories[1])
    assert pos in categories, (
        f'{variable}: pinned reporting category {pos!r} is absent from this sample {categories}'
    )

    table = np.array([
        [int((x == pos).sum()), int((x != pos).sum())],
        [int((y == pos).sum()), int((y != pos).sum())],
    ])
    chi2, p_chi2, dof, expected = chi2_contingency(table, correction=True)
    n   = table.sum()
    phi = float(np.sqrt(chi2 / n))

    if (expected < 5).any():
        _, p = fisher_exact(table)
        test = 'fisher_exact'
    else:
        p = p_chi2
        test = 'chi2'

    return dict(
        task=task, variable=variable, var_type='binary',
        n_normative_full=n_normative_full, n_included=n_inc, n_excluded=n_exc,
        included_center=float((x == pos).mean()), included_spread=None,
        excluded_center=float((y == pos).mean()), excluded_spread=None,
        test=test, stat=float(chi2), p_value=float(p), effect_size=phi,
        significant=bool(p < 0.05), suppressed=False,
        positive_category=str(pos),
    )


def _balance_row_label(row):
    """Binary rows name the category their proportion refers to."""
    if row['var_type'] == 'binary' and row.get('positive_category') is not None:
        return f'{row["variable"]} (% {row["positive_category"]})'
    return row['variable']


def _print_balance_row(row):
    label = _balance_row_label(row)
    if row['suppressed']:
        print(f'  {label:<26} SUPPRESSED '
              f'(n_included={row["n_included"]} or n_excluded={row["n_excluded"]} < 20)')
        return
    star = ' *' if row['p_value'] < 0.05 else ''
    if row['var_type'] == 'continuous':
        inc_s = f'{row["included_center"]:.3f}±{row["included_spread"]:.3f}'
        exc_s = f'{row["excluded_center"]:.3f}±{row["excluded_spread"]:.3f}'
    else:
        inc_s = f'{row["included_center"]:.1%}'
        exc_s = f'{row["excluded_center"]:.1%}'
    print(f'  {label:<26} incl={inc_s:<18} excl={exc_s:<18}  '
          f'{row["test"]:<12} stat={row["stat"]:.3f}  ES={row["effect_size"]:+.3f}  '
          f'p={row["p_value"]:.4g}{star}')


CONTINUOUS_SPECS = [
    ('D',                  lambda task: 'D'),
    ('age_at_etm',         lambda task: f'age_at_etm_{task}'),
    ('education_ordinal',  lambda task: 'education_ordinal'),
    ('bmi',                lambda task: f'bmi_{task}'),
    ('cci',                lambda task: f'cci_{task}'),
]
BINARY_SPECS = [
    ('sex',   lambda task: 'sex'),
    ('touch', lambda task: f'touch_{task}'),
]
BALANCE_POOL_SIZE = len(ETM_TASKS) * (len(CONTINUOUS_SPECS) + len(BINARY_SPECS))

balance_rows = []
for task in ETM_TASKS:
    ab_mdd, c_mdd = build_mdd_normative_and_fitbit(task, cohort, wf)
    included_ids  = set(c_mdd['person_id'])
    included      = ab_mdd[ab_mdd['person_id'].isin(included_ids)]
    excluded      = ab_mdd[~ab_mdd['person_id'].isin(included_ids)]
    n_normative_full = len(ab_mdd)

    print(f'\n{"─" * 72}')
    print(f'  {task.upper()}  ·  MDD Normative N={n_normative_full:,}  '
          f'→  Fitbit-included N={len(included):,} / Fitbit-excluded N={len(excluded):,}')
    print(f'{"─" * 72}')

    for label, col_fn in CONTINUOUS_SPECS:
        col = col_fn(task)
        row = _continuous_balance_row(task, label, included[col], excluded[col], n_normative_full)
        balance_rows.append(row)
        _print_balance_row(row)

    for label, col_fn in BINARY_SPECS:
        col = col_fn(task)
        row = _binary_balance_row(task, label, included[col], excluded[col], n_normative_full)
        balance_rows.append(row)
        _print_balance_row(row)


In [ ]:
assert len(balance_rows) == BALANCE_POOL_SIZE, (
    f'Expected {BALANCE_POOL_SIZE} balance rows (7 variables x 4 tasks); got '
    f'{len(balance_rows)}. Sub-20 cells must emit a suppressed placeholder row, not be omitted.'
)

balance_df = pd.DataFrame(balance_rows)

col_order = [
    'task', 'variable', 'var_type',
    'n_normative_full', 'n_included', 'n_excluded',
    'included_center', 'included_spread', 'excluded_center', 'excluded_spread',
    'test', 'stat', 'p_value', 'significant', 'effect_size', 'suppressed',
    'positive_category',
]
balance_df = balance_df[col_order].reset_index(drop=True)

print('\n' + '=' * 100)
print('  MDD NORMATIVE vs. FITBIT-ELIGIBLE BALANCE SUMMARY  (nominal p < 0.05, no correction)')
print('=' * 100)
n_supp_bal   = int(balance_df['suppressed'].sum())
n_sig_bal    = int(balance_df['significant'].fillna(False).sum())
n_tested_bal = int((~balance_df['suppressed'].fillna(False)).sum())
print(f'Nominally significant imbalances: {n_sig_bal} / {n_tested_bal} non-suppressed tests  '
      f'({n_supp_bal} rows suppressed, n < 20)')
print('\nNominal p, no correction.')

save_res(RES_OUT, 'mdd_normative_vs_fitbit_balance.csv',  OVERWRITE, balance_df,  file_type='csv')
save_parquet(balance_df, 'results/mdd_normative_vs_fitbit_balance.parquet')

`*_center`: mean, or the proportion female / touchscreen for binary rows; `*_spread`: SD;
`effect_size`: Cohen's d (included − excluded) for continuous rows; phi for binary rows,
which is unsigned (the direction is given by `*_center`).

## Task 5: Antipsychotic Exclusion

The normative-deviation analysis (`D ~ 0 + C(group)`, HC3) refitted after excluding MDD and BD
participants with an antipsychotic prescription active at that task's test date
(`on_antipsychotic_{task}`, from `02_covariates`; phenothiazine antiemetics are not counted).
The HC model and `D` are unchanged. Each task is also refitted on the full sample, which
reproduces `04a`, so the change in each estimate can be read directly.

In [ ]:
# ── Task 5 — normative-deviation refit, antipsychotic-excluded ───────────────
AP_COL_TMPL = 'on_antipsychotic_{task}'

# n_prior_episodes_{task} is nullable-Int64 in cohort_master; scipy needs float.
for _t in ETM_TASKS:
    cohort[f'n_prior_episodes_{_t}'] = pd.to_numeric(
        cohort[f'n_prior_episodes_{_t}'], errors='coerce').astype(float)


def build_normative_samples(task, cohort):
    """MDD and BD Normative samples with D and an on-antipsychotic flag for `task`.

    Returns the full samples so Task 5b can reuse them without refitting anything."""
    m_norm, sigma_hc, hc = fit_normative_model(cohort, task)

    def _norm_sample(group):
        sub = normative_sample(cohort, task, group, m_norm, sigma_hc)
        # 1.0 = active antipsychotic at this task's EtM date; 0.0 = not; NaN only for HC.
        sub['on_ap'] = sub[AP_COL_TMPL.format(task=task)].fillna(0.0) == 1.0
        return sub

    return dict(
        n_hc=len(hc), sigma_hc=sigma_hc, r2_hc=m_norm.rsquared,
        mdd=_norm_sample('MDD'), bd=_norm_sample('BD'),
    )


noap_results  = []
noap_samples  = {}      # task -> build_normative_samples() output, reused by Task 5b

for task in ETM_TASKS:
    s = build_normative_samples(task, cohort)
    noap_samples[task] = s

    full_mdd, full_bd = s['mdd'], s['bd']
    keep_mdd, drop_mdd = full_mdd[~full_mdd['on_ap']], full_mdd[full_mdd['on_ap']]
    keep_bd,  drop_bd  = full_bd[~full_bd['on_ap']],   full_bd[full_bd['on_ap']]

    print(f'\n{"─" * 78}')
    print(f'  {task.upper()}  ·  outcome: {TASK_CONFIG[task]["outcome"]}')
    print(f'{"─" * 78}')
    print(f'HC model   N={s["n_hc"]:,}  R²={s["r2_hc"]:.3f}  σ_HC={s["sigma_hc"]:.4f}  '
          f'(unchanged — HC has no medication data)')
    print(f'MDD  Normative N={len(full_mdd):,}  →  dropped {len(drop_mdd):,} on antipsychotic  '
          f'→  retained N={len(keep_mdd):,}')
    print(f'BD   Normative N={len(full_bd):,}  →  dropped {len(drop_bd):,} on antipsychotic  '
          f'→  retained N={len(keep_bd):,}')

    m_full, ct_full = fit_group_model(full_mdd, full_bd)
    m_noap, ct_noap = fit_group_model(keep_mdd, keep_bd)

    _spec = [
        ('MDD',      'C(group)[MDD]', len(keep_mdd), None,          len(full_mdd), None),
        ('BD',       'C(group)[BD]',  None,          len(keep_bd),  None,          len(full_bd)),
        ('contrast', None,            len(keep_mdd), len(keep_bd),  len(full_mdd), len(full_bd)),
    ]
    for grp, param, n_mdd, n_bd, n_mdd_full, n_bd_full in _spec:
        n_small = min([n for n in (n_mdd, n_bd) if n is not None])
        if n_small < MIN_CELL:
            print(f'  {grp}: retained N={n_small} < {MIN_CELL}, suppressing')
            noap_results.append(suppressed_row(
                task=task, analysis='Normative_noAP', phenotype=None,
                group=grp, n_hc=s['n_hc'], n_mdd=n_mdd, n_bd=n_bd,
            ))
        elif param is None:   # BD − MDD contrast
            noap_results.append(result_row(
                task=task, analysis='Normative_noAP', phenotype=None,
                group=grp, n_hc=s['n_hc'], n_mdd=n_mdd, n_bd=n_bd,
                beta=float(ct_noap.effect), se=float(ct_noap.sd),
                t=float(ct_noap.tvalue),    p=float(ct_noap.pvalue),
                effect_size=float(ct_noap.effect), r2=None,
            ))
        else:
            noap_results.append(result_row(
                task=task, analysis='Normative_noAP', phenotype=None,
                group=grp, n_hc=s['n_hc'], n_mdd=n_mdd, n_bd=n_bd,
                beta=m_noap.params[param], se=m_noap.bse[param],
                t=m_noap.tvalues[param],   p=m_noap.pvalues[param],
                effect_size=m_noap.params[param], r2=None,
            ))

        # Full-sample reference; reproduces 04a (checked against model_results in 05).
        if param is None:
            b_f, se_f, p_f = float(ct_full.effect), float(ct_full.sd), float(ct_full.pvalue)
        else:
            b_f, se_f, p_f = m_full.params[param], m_full.bse[param], m_full.pvalues[param]
        noap_results[-1].update(
            beta_full=float(b_f), se_full=float(se_f), p_full=float(p_f),
            n_mdd_full=n_mdd_full, n_bd_full=n_bd_full,
            n_dropped_mdd=len(drop_mdd) if n_mdd is not None else None,
            n_dropped_bd=len(drop_bd)   if n_bd  is not None else None,
        )

        row = noap_results[-1]
        if row['suppressed']:
            continue
        star  = ' *' if row['p_value'] < 0.05 else ''
        delta = row['beta'] - row['beta_full']
        # Shift in units of the full-sample SE
        rel   = abs(delta) / row['se_full'] if row['se_full'] else float('nan')
        print(f'  {grp:<9} full β={row["beta_full"]:+.3f} (p={row["p_full"]:.4g})   '
              f'no-AP β={row["beta"]:+.3f} [{row["ci_lo"]:+.3f},{row["ci_hi"]:+.3f}] '
              f'(p={row["p_value"]:.4g}){star}   Δβ={delta:+.3f} ({rel:.2f}×SE_full)')

In [ ]:
NOAP_POOL_SIZE = len(ETM_TASKS) * 3   # MDD / BD / contrast per task

assert len(noap_results) == NOAP_POOL_SIZE, (
    f'Expected {NOAP_POOL_SIZE} antipsychotic-sensitivity results (MDD/BD/contrast x 4 tasks); '
    f'got {len(noap_results)}. Sub-20 cells must emit a suppressed placeholder row, not be '
    'omitted.'
)

noap_df = pd.DataFrame(noap_results)

# Nominal p, no correction
noap_df['significant'] = noap_df['p_value'] < 0.05
noap_df['delta_beta']  = noap_df['beta'] - noap_df['beta_full']

col_order = [
    'task', 'analysis', 'phenotype', 'group',
    'n_hc', 'n_mdd', 'n_bd', 'n_mdd_full', 'n_bd_full', 'n_dropped_mdd', 'n_dropped_bd',
    'beta', 'se_hc3', 'ci_lo', 'ci_hi',
    'beta_full', 'se_full', 'p_full', 'delta_beta',
    't_stat', 'p_value', 'p_bh', 'significant',
    'effect_size', 'r2', 'suppressed',
]
noap_df = noap_df[col_order].reset_index(drop=True)

print('\n' + '=' * 122)
print('  ANTIPSYCHOTIC-EXCLUSION SENSITIVITY — normative-deviation refit')
print('  (nominal p < 0.05, NO multiple-testing correction; detached from the main 28-test pool)')
print('=' * 122)
print(f'{"Task":<18} {"Group":<9} {"N_kept":>14} {"dropped":>9} '
      f'{"β_full":>8} {"β_noAP":>8} {"Δβ":>7} {"95% CI (noAP)":>18} '
      f'{"p_full":>9} {"p_noAP":>9} {"sig":>4}')
print('-' * 122)
for _, r in noap_df.iterrows():
    if r['group'] == 'contrast':
        n_kept = f'{int(r["n_mdd"]):,}/{int(r["n_bd"]):,}'
        n_drop = f'{int(r["n_dropped_mdd"]):,}/{int(r["n_dropped_bd"]):,}'
    elif r['group'] == 'MDD':
        n_kept, n_drop = f'{int(r["n_mdd"]):,}', f'{int(r["n_dropped_mdd"]):,}'
    else:
        n_kept, n_drop = f'{int(r["n_bd"]):,}',  f'{int(r["n_dropped_bd"]):,}'
    if r['suppressed']:
        print(f'{r["task"]:<18} {r["group"]:<9} {n_kept:>14} {n_drop:>9}   SUPPRESSED (n < 20)')
        continue
    ci = f'[{r["ci_lo"]:+.3f},{r["ci_hi"]:+.3f}]'
    print(f'{r["task"]:<18} {r["group"]:<9} {n_kept:>14} {n_drop:>9} '
          f'{r["beta_full"]:>+8.3f} {r["beta"]:>+8.3f} {r["delta_beta"]:>+7.3f} {ci:>18} '
          f'{r["p_full"]:>9.4g} {r["p_value"]:>9.4g} {"*" if r["significant"] else "":>4}')

print('-' * 122)
n_sig   = int(noap_df['significant'].fillna(False).sum())
n_test  = int((~noap_df['suppressed'].fillna(False)).sum())
max_shift = noap_df['delta_beta'].abs().max()
print(f'Nominally significant: {n_sig} / {n_test} non-suppressed tests. '
      f'Largest |Δβ| vs. the full sample: {max_shift:.3f}.')
print('\nβ is Glass\'s Δ; β_full is the full-sample refit, which reproduces 04a.')

save_res(RES_OUT, 'antipsychotic_sensitivity.csv', OVERWRITE, noap_df, file_type='csv')
save_parquet(noap_df, 'results/antipsychotic_sensitivity.parquet')

## Task 5b: Excluded vs Retained Participants

Within MDD and BD, compares the participants excluded in Task 5 with those retained, on `D`,
age, education, BMI, CCI, number of prior episodes, sex and device type (same tests as Task 4).
If those excluded were more impaired, a change in the Task 5 estimates could reflect illness
severity rather than medication.

In [ ]:
# ── Task 5b — dropped vs retained, within each group ─────────────────────────
AP_CONTINUOUS_SPECS = [
    ('D',                  lambda task: 'D'),
    ('age_at_etm',         lambda task: f'age_at_etm_{task}'),
    ('education_ordinal',  lambda task: 'education_ordinal'),
    ('bmi',                lambda task: f'bmi_{task}'),
    ('cci',                lambda task: f'cci_{task}'),
    ('n_prior_episodes',   lambda task: f'n_prior_episodes_{task}'),
]
AP_BINARY_SPECS = [
    ('sex',   lambda task: 'sex'),
    ('touch', lambda task: f'touch_{task}'),
]
AP_BALANCE_POOL_SIZE = (
    len(ETM_TASKS) * 2 * (len(AP_CONTINUOUS_SPECS) + len(AP_BINARY_SPECS))
)

_AP_RENAME = {
    'n_normative_full': 'n_normative',
    'n_included':       'n_dropped',   'n_excluded':       'n_retained',
    'included_center':  'dropped_center', 'included_spread': 'dropped_spread',
    'excluded_center':  'retained_center', 'excluded_spread': 'retained_spread',
}


def _binary_balance_row_safe(task, variable, dropped, retained, n_norm):
    """_binary_balance_row, but returns a suppressed row instead of failing when the
    variable has only one category in this sample."""
    cats = set(dropped.dropna().unique()) | set(retained.dropna().unique())
    if len(cats) != 2:
        return dict(
            task=task, variable=variable, var_type='binary',
            n_normative_full=n_norm, n_included=int(dropped.notna().sum()),
            n_excluded=int(retained.notna().sum()),
            included_center=None, included_spread=None,
            excluded_center=None, excluded_spread=None,
            test=None, stat=None, p_value=None, effect_size=None,
            significant=None, suppressed=True,
            positive_category=BINARY_POS_CATEGORY.get(variable),
        )
    return _binary_balance_row(task, variable, dropped, retained, n_norm)


def _print_ap_balance_row(row):
    """As _print_balance_row, with the two sides labelled dropped / retained."""
    label = _balance_row_label(row)
    if row['suppressed']:
        print(f'  {label:<26} SUPPRESSED '
              f'(n_dropped={row["n_included"]} or n_retained={row["n_excluded"]} < 20, '
              f'or variable not binary in this sample)')
        return
    star = ' *' if row['p_value'] < 0.05 else ''
    if row['var_type'] == 'continuous':
        d_s = f'{row["included_center"]:.3f}±{row["included_spread"]:.3f}'
        k_s = f'{row["excluded_center"]:.3f}±{row["excluded_spread"]:.3f}'
    else:
        d_s = f'{row["included_center"]:.1%}'
        k_s = f'{row["excluded_center"]:.1%}'
    flag = ''
    if row.get('dropped_more_impaired') is True:
        flag = '  <-- dropped are MORE impaired'
    elif row.get('dropped_more_impaired') is False:
        flag = '  <-- dropped are LESS impaired'
    print(f'  {label:<26} drop={d_s:<18} keep={k_s:<18}  '
          f'{row["test"]:<12} ES={row["effect_size"]:+.3f}  '
          f'p={row["p_value"]:.4g}{star}{flag}')


ap_balance_rows = []
for task in ETM_TASKS:
    higher_better = TASK_CONFIG[task]['higher_is_better']
    for grp in ['MDD', 'BD']:
        full    = noap_samples[task]['mdd' if grp == 'MDD' else 'bd']
        dropped = full[full['on_ap']]
        retained = full[~full['on_ap']]
        n_norm  = len(full)

        print(f'\n{"─" * 78}')
        print(f'  {task.upper()}  ·  {grp}  ·  Normative N={n_norm:,}  →  '
              f'dropped N={len(dropped):,} / retained N={len(retained):,}')
        print(f'{"─" * 78}')

        for label, col_fn in AP_CONTINUOUS_SPECS:
            col = col_fn(task)
            row = _continuous_balance_row(task, label, dropped[col], retained[col], n_norm)
            row['group'] = grp
            # D only: is the dropped group on the impaired side, given the task's direction?
            if label == 'D' and not row['suppressed']:
                worse = (row['included_center'] < row['excluded_center']) if higher_better \
                        else (row['included_center'] > row['excluded_center'])
                row['dropped_more_impaired'] = bool(worse)
            else:
                row['dropped_more_impaired'] = None
            ap_balance_rows.append(row)
            _print_ap_balance_row(row)

        for label, col_fn in AP_BINARY_SPECS:
            col = col_fn(task)
            row = _binary_balance_row_safe(task, label, dropped[col], retained[col], n_norm)
            row['group'] = grp
            row['dropped_more_impaired'] = None
            ap_balance_rows.append(row)
            _print_ap_balance_row(row)

In [ ]:
assert len(ap_balance_rows) == AP_BALANCE_POOL_SIZE, (
    f'Expected {AP_BALANCE_POOL_SIZE} rows (8 variables x 4 tasks x 2 groups); got '
    f'{len(ap_balance_rows)}. Sub-20 cells must emit a suppressed placeholder row.'
)

ap_balance_df = pd.DataFrame(ap_balance_rows).rename(columns=_AP_RENAME)

ap_col_order = [
    'task', 'group', 'variable', 'var_type',
    'n_normative', 'n_dropped', 'n_retained',
    'dropped_center', 'dropped_spread', 'retained_center', 'retained_spread',
    'test', 'stat', 'p_value', 'significant', 'effect_size',
    'dropped_more_impaired', 'suppressed', 'positive_category',
]
ap_balance_df = ap_balance_df[ap_col_order].reset_index(drop=True)

print('\n' + '=' * 104)
print('  ANTIPSYCHOTIC DROPPED-vs-RETAINED BALANCE  (nominal p < 0.05, no correction)')
print('=' * 104)
n_supp = int(ap_balance_df['suppressed'].sum())
n_sig  = int(ap_balance_df['significant'].fillna(False).sum())
n_test = int((~ap_balance_df['suppressed'].fillna(False)).sum())
print(f'Nominally significant imbalances: {n_sig} / {n_test} non-suppressed tests  '
      f'({n_supp} rows suppressed, n < 20)')

print('\n--- The D rows: were the dropped participants more impaired? ---')
_d = ap_balance_df[(ap_balance_df['variable'] == 'D') & ~ap_balance_df['suppressed']]
print(f'{"task":<18} {"group":<6} {"D_dropped":>11} {"D_retained":>11} {"Cohen d":>8} '
      f'{"p":>9} {"dropped more impaired?":>24}')
for _, r in _d.iterrows():
    print(f'{r["task"]:<18} {r["group"]:<6} {r["dropped_center"]:>+11.3f} '
          f'{r["retained_center"]:>+11.3f} {r["effect_size"]:>+8.3f} {r["p_value"]:>9.4g} '
          f'{str(r["dropped_more_impaired"]):>24}')

print('\nNominal p, no correction.')

save_res(RES_OUT, 'antipsychotic_dropped_vs_retained_balance.csv', OVERWRITE,
         ap_balance_df, file_type='csv')
save_parquet(ap_balance_df, 'results/antipsychotic_dropped_vs_retained_balance.parquet')

Columns: Task 5, `beta` and `p_value` after exclusion, `beta_full` and `p_full` on the full
sample, `delta_beta = beta − beta_full`; Task 5b, `effect_size` is Cohen's d (excluded − retained) or unsigned phi, and
`dropped_more_impaired` (`D` rows only) takes each task's direction into account.